# Reddit Suspicious User ID Stripping 

This notebook will strip the IDs of all suspicious users in posts that do not mention bots. These IDs need to be collected so they can be connected to other posts and comments they have been involved in.

In [ ]:
# ---- Libraries -----------------------------------------------------------------
# Core data handling.
import pandas as pd
import numpy as np
import os

# Show every column/row when inspecting a frame - project convention.
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

import re                                  # username pattern matching
from string import punctuation
import functools
import operator
from dotenv import load_dotenv             # loads the Reddit credentials file
from pmaw import PushshiftAPI              # Pushshift wrapper (unused in this notebook)
import praw                                # Reddit API client
import prawcore                            # praw's exception types
from prawcore.exceptions import NotFound   # raised for deleted / nonexistent accounts
import pprint
import ast
import pprint
import os
import itertools
import pickle

In [ ]:
# Source corpus: every r/TheseFuckingAccounts submission, one row per post.
csv='/data/Emily_data/TheseFuckingAccounts_submissions.csv'

In [ ]:
# Read the raw submissions. NOTE: this file has 124 mixed-dtype columns, so pandas emits a
# DtypeWarning here; pass low_memory=False to silence it and get consistent column types.
subs=pd.read_csv(csv)

Dropping NA values from selftext which is the post content. We checked for duplicates of posts in previous notebook and know there aren't any so we don't need to do that here. 

In [ ]:
# Keep only posts that have body text - the username mentions live in selftext.
# This drops link posts, which carry no accusation text to parse.
posts=subs.dropna(subset=['selftext'])

Remove bots

In [ ]:
# Exclude posts about *bots* so that what remains is accusations against human-run accounts.
# "(?i)" makes the match case-insensitive.
# CAUTION: this is a substring match, so it also removes posts containing "robot", "botany",
# "sandbot" etc., and it drops the whole post even if "bot" appears only in passing.
# It also emits a UserWarning about match groups in the pattern.
posts=posts[~posts.selftext.str.contains("(?i)bot")]
posts=posts[~posts.title.str.contains("(?i)bot")]

In [ ]:
# Drop moderation placeholders: these posts have no recoverable body text.
remove = ['[removed]','[deleted]']

posts=posts.loc[posts.selftext.isin(remove) == False]

In [ ]:
# Match a username written as "u/name" or "user/name" and capture just the name part.
# CAUTION: with no left boundary this also matches the tail of unrelated paths, e.g.
# "imgur.com/gallery/AbCdE/comment/123" yields "comment". It also makes no accused/victim
# distinction, so "copied from u/original" collects the victim too.
user_pat = "(?:u/|user/)([A-Za-z0-9_-]+)"

def find_text(df, col_name, pat, new_col):
    """Add `new_col` holding the list of all `pat` matches found in `col_name`."""
    # .apply(str.lower) normalises case first, so names are collected lowercased.
    df[new_col] = df[col_name].apply(str.lower).str.findall(pat)
    return df

# Collect mentions from the body and the title separately, then concatenate the two lists
# so each post carries one combined list of every username it names.
posts = find_text(posts, 'selftext', user_pat, 'users1')
posts = find_text(posts, 'title', user_pat, 'users2')
posts['users'] = posts['users1'] + posts['users2']

# get first appearance date per username
# explode() turns the per-post list of names into one row per (post, name), after which
# groupby().min() gives the earliest time each name was mentioned anywhere in the subreddit.
first_appearance = (posts[['users', 'created_utc']]
                    .explode('users')
                    .rename(columns={'users': 'username'})
                    .dropna(subset=['username'])       # posts that matched nothing
                    .query("username != ''")           # guard against empty captures
                    .groupby('username')['created_utc']
                    .min()
                    .reset_index()
                    .rename(columns={'created_utc': 'first_appearance_utc'})
                   )

In [ ]:
# Save the name -> first-mention table. Commented out so a re-run does not overwrite the
# saved artifact by accident; uncomment deliberately when regenerating it.
#first_appearance.to_parquet('Names_and_first_mention.parquet', engine='pyarrow', compression='zstd')

## API call setup 

In [ ]:
# IPython extension that lets the %dotenv magic below read a credentials file.
%load_ext dotenv

In [ ]:
# Load the Reddit API credentials into the environment. This file lives outside the repo
# so the keys are never committed.
%dotenv /home/eeporter/Reddit_project1_notebooks/reddit_env_vars.env

In [ ]:
# Authenticate against the Reddit API with a script-type app.
# ratelimit_seconds=360: on a 429, wait up to 6 minutes for the window to reset rather
# than raising immediately.
# NOTE: the credentials file defines the login key as `login_name`, not `user_name`, so
# the username below currently resolves to None. Change to os.getenv("login_name") if
# this notebook is re-run and an authenticated (non-read-only) session is needed.
reddit_api = praw.Reddit(
client_id=os.getenv("praw_script"),
client_secret=os.getenv("praw_key"),
username=os.getenv("user_name"),
password=os.getenv("password"),
user_agent= os.getenv("user_agent"),ratelimit_seconds=360)

## Collect User ID and Status 

In [ ]:
def account_data(list_of_unique_usernames):
    """
    Look up every username against the Reddit API and record its state plus raw profile.

    Returns {username: {'Active'|'Suspended'|'Deleted': profile_dict_or_'No Data'}}.
    Each account is retried up to 10 times on rate-limit errors.
    """
    user_data={}
    for x in list_of_unique_usernames:
        #init of dict with x as key[0]
        user_data[x]={}
        #retries for failure
        retry_count = 0
        while retry_count < 10:
            try:
                #get info on account
                # Note: praw is lazy here - the HTTP request fires on first attribute access below.
                redditor_info = reddit_api.redditor(str(x))
                #print(f"Fetching data for: {redditor_info.name}")

                try:
                    #check attributes profiles have diff meta data depending on status
                    # A suspended account returns a stub carrying is_suspended;
                    # a live account carries an id; anything else is treated as deleted.
                    if hasattr(redditor_info, 'is_suspended') == True:
                        y='Suspended'
                    elif hasattr(redditor_info, 'id') == True:
                        y='Active'
                    else:
                        y='Deleted'
                except prawcore.exceptions.NotFound:
                    # Deleted or never-existed account: the lazy fetch 404s here.
                    y='Deleted'
                #if user is still active grab and nest data
                # __dict__ is the full profile payload, kept for later attribute analysis.
                if y == 'Active' or y == 'Suspended':
                    #print('Yes')
                    user_data[x][y]=redditor_info.__dict__
                #if not active just record it
                else:
                    user_data[x][y]='No Data'
                    #print('None')
            #handle errors
            except prawcore.exceptions.TooManyRequests as e:
                # Rate limited: count the attempt and loop round; give up after 10 tries.
                print(f"Querying user {x} failed with {e} retry count currently {retry_count}")
                retry_count += 1
                if retry_count == 10:
                    raise e
            except Exception as e:
                # Anything unexpected is fatal - better to stop than to record bad states.
                print (f"Hit unexpected error with class {e.__class__}:{e}")
                raise e
            else:
                # Cheeky way to bypass doing this work again
                # (no exception raised, so force-exit the retry loop for this user)
                retry_count = 10

    return user_data

In [ ]:
# Run the lookup over every extracted username. This is the slow step: one API call per
# account, so expect it to run for hours on a full name list.
user_dict=account_data(first_appearance['username'])

In [ ]:
def dict_to_dfs(nested_dict):
    """transform nested dict into separate dfs based
    on outcome since available data is different for
    each subgroup."""
    #level one keys are user names so those become the first col
    df=pd.DataFrame(nested_dict.keys(),columns=['User_Names'])
    #make some empty lists to add as cols later
    outcome_list=[]
    user_data_dicts=[]
    #dig into the nested dict and append values to empty list
    # Each user maps to exactly one {outcome: payload} pair, so this inner loop runs once
    # per user and the two lists stay aligned with the User_Names column.
    for key1,value1 in nested_dict.items():
        for key2,value2 in value1.items():
            outcome_list.append(key2)
            user_data_dicts.append(value2)
    #normalize dict to df format and drop object cols

    df['User_Data_Dict']=user_data_dicts
    # These three cannot be written to parquet: the raw dict column plus the two live praw
    # objects (_reddit, subreddit) are not serialisable, so drop them after flattening.
    drop_cols=['User_Data_Dict','_reddit','subreddit']
    # json_normalize turns each nested profile payload into flat columns.
    df=df.join(pd.json_normalize(df['User_Data_Dict'])).drop(drop_cols, axis='columns')
    df['Account_Outcome']=outcome_list

    return   df

### Change column drop and save to functional format! 

In [ ]:
# Flatten the nested lookup results into one row per account.
user_df=dict_to_dfs(user_dict)

In [ ]:
# Inspect the flattened profile columns before merging.
user_df.columns

In [ ]:
# Inspect the first-appearance columns to confirm the merge keys line up.
first_appearance.columns

In [ ]:
# Attach each account's profile/status to its first-mention date.
# how='inner' keeps only names that were successfully looked up, dropping any the API
# never resolved.
# FIXED: this line previously ended in a stray "." after the closing paren, which made the
# cell a SyntaxError and stopped the notebook here.
merged_df = pd.merge(first_appearance, user_df, left_on='username', right_on='User_Names', how='inner')

In [ ]:
# 'username' and 'User_Names' are the same value from either side of the merge; keep one.
account_profiles=merged_df.drop(columns='User_Names')

In [ ]:
# Save the account profiles. Commented out so a re-run does not overwrite the saved
# artifact by accident; uncomment deliberately when regenerating it.
#account_profiles.to_parquet('User_Profiles.parquet', engine='pyarrow', compression='zstd')